In [4]:
# check if all jars are available
!ls /home/jovyan/.ivy2/jars

ls: cannot access '/home/jovyan/.ivy2/jars': No such file or directory


In [20]:
from pyspark.sql import SparkSession

base_bucket_basic = "s3a://client-bucket/"
base_bucket = "s3a://client-bucket/topics/fullfillment.TYROK.TYROK.client/"
meta_bucket = "s3a://tyrok-bucket/"
file = "s3a://client-bucket/topics/fullfillment.TYROK.TYROK.client/year=2026/month=02/day=28/hour=15/fullfillment.TYROK.TYROK.client+0+0000000000.snappy.parquet"
file_star = "s3a://client-bucket/topics/fullfillment.TYROK.TYROK.client/year=2026/month=02/day=28/hour=15/*.parquet"


In [21]:
# removed configs
#.config("spark.sql.shuffle.partitions", "2") \

In [22]:
# Configuration alignée sur votre fichier YAML (Master: 7077, MinIO: admin/password123)
# sandaga_catalog is the path-based catalog defined above: .config("spark.sql.catalog.sandaga_catalog", "org.apache.iceberg.spark.SparkCatalog")
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Iceberg-MinIO-Parquet") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.3.1,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.sandaga_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.sandaga_catalog.type", "hadoop") \
    .config("spark.sql.catalog.sandaga_catalog.warehouse", meta_bucket+"warehouse") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.iceberg.vectorization.enabled", "false") \
    .config("spark.sql.iceberg.distribution-mode", "hash") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("✅ Session Spark initialisée avec Iceberg !")


✅ Session Spark initialisée avec Iceberg !


In [23]:
#==== Namespace et base de données ====
# Créer une base de données dans ce catalogue
spark.sql("CREATE DATABASE IF NOT EXISTS sandaga_catalog.db")

DataFrame[]

In [24]:
# checks
spark.sql("SHOW NAMESPACES IN sandaga_catalog").show()

+---------+
|namespace|
+---------+
|       db|
+---------+



In [25]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS sandaga_catalog.db.client (
        id bigint,
        code string,
        name string,
        actif int,
        year string,
        month string,
        day string,
        hour string
    )
    USING iceberg
    PARTITIONED BY (year, month, day, hour)
""")

DataFrame[]

In [26]:
spark.sql("""
    SHOW TABLES IN sandaga_catalog.db
""").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|       db|   client|      false|
+---------+---------+-----------+



In [27]:
#Voir les données (table vide car pas de snapshot ou ajout de fichier parquet dans les metadonnées Iceberg)
spark.sql("""
SELECT * 
FROM sandaga_catalog.db.client
ORDER BY id
""").show()

+---+----+-----------------+-----+----+-----+---+----+
| id|code|             name|actif|year|month|day|hour|
+---+----+-----------------+-----+----+-----+---+----+
|  1|C001| Entreprise FIRST|    1|2026|   02| 28|  15|
|  2|C002|Entreprise SECOND|    1|2026|   02| 28|  15|
|  3|C003| Entreprise THIRD|    1|2026|   02| 28|  15|
|  4|C004|Entreprise FOURTH|    1|2026|   02| 28|  15|
|  5|C005| Entreprise FIFTH|    1|2026|   02| 28|  15|
|  6|C006| Entreprise SIXTH|    1|2026|   02| 28|  15|
|  7|C007|     Entreprise 7|    1|2026|   02| 28|  15|
|  8|C008|     Entreprise 8|    1|2026|   02| 28|  15|
+---+----+-----------------+-----+----+-----+---+----+



In [ ]:
#==== Snapshot ====
# On crée la table Iceberg à la racine de vos données existantes
# cronner pour importer de nouvelles données
spark.sql(f"""
    CALL sandaga_catalog.system.add_files(
        table => 'sandaga_catalog.db.client',
        source_table => 'parquet.`{base_bucket_basic}`'
    )
""")


In [29]:
#Voir la liste des fichiers physiques indexés par Iceberg
spark.sql("""
SELECT file_path, partition, file_size_in_bytes 
FROM sandaga_catalog.db.client.files
""").show()

+--------------------+------------------+------------------+
|           file_path|         partition|file_size_in_bytes|
+--------------------+------------------+------------------+
|s3a://client-buck...|{2026, 02, 28, 15}|              1375|
|s3a://client-buck...|{2026, 02, 28, 15}|              1377|
+--------------------+------------------+------------------+



In [37]:
#Voir les données
spark.sql("""
SELECT * 
FROM sandaga_catalog.db.client
ORDER BY id
""").show()

+---+----+-----------------+-----+----+-----+---+----+
| id|code|             name|actif|year|month|day|hour|
+---+----+-----------------+-----+----+-----+---+----+
|  1|C001| Entreprise FIRST|    1|2026|   02| 28|  15|
|  2|C002|Entreprise SECOND|    1|2026|   02| 28|  15|
|  3|C003| Entreprise THIRD|    1|2026|   02| 28|  15|
|  4|C004|Entreprise FOURTH|    1|2026|   02| 28|  15|
|  5|C005| Entreprise FIFTH|    1|2026|   02| 28|  15|
|  6|C006| Entreprise SIXTH|    1|2026|   02| 28|  15|
|  7|C007|     Entreprise 7|    1|2026|   02| 28|  15|
|  8|C008|     Entreprise 8|    1|2026|   02| 28|  15|
+---+----+-----------------+-----+----+-----+---+----+



In [38]:
db_name="sandaga_catalog"
schema_name="db"
table_name="client"

In [33]:
# cronner (compression)
spark.sql(f"""
    CALL {db_name}.system.rewrite_data_files(
        table => '{db_name}.{schema_name}.{table_name}',
        strategy => 'binpack',
        options => map('target-file-size-bytes', '134217728') -- 128 Mo
    )
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

In [35]:
# triez frequement vos données physiquement.
#spark.sql(f"ALTER TABLE {db_name}.{schema_name}.{table_name} WRITE ORDERED BY id")

spark.sql(f"""
CALL {db_name}.system.rewrite_data_files(
  table => '{db_name}.{schema_name}.{table_name}', 
  strategy => 'sort', 
  sort_order => 'id ASC'
)
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint, failed_data_files_count: int]

In [36]:
# suppression des snapshots
spark.sql(f"CALL {db_name}.system.expire_snapshots('{db_name}.{schema_name}.{table_name}')")

DataFrame[deleted_data_files_count: bigint, deleted_position_delete_files_count: bigint, deleted_equality_delete_files_count: bigint, deleted_manifest_files_count: bigint, deleted_manifest_lists_count: bigint, deleted_statistics_files_count: bigint]

In [39]:
# nombre de fichier par jour
result = spark.sql(f"""
    SELECT year, month, day, hour, count(*) as count
    FROM {db_name}.{schema_name}.{table_name}
    GROUP BY year, month, day, hour
    ORDER BY year, month, day, hour
""")
result.show()

+----+-----+---+----+-----+
|year|month|day|hour|count|
+----+-----+---+----+-----+
|2026|   02| 28|  15|    8|
+----+-----+---+----+-----+



In [40]:
# Exécuter une requête SQL avec un filtre sur la table Iceberg
result_filtered = spark.sql(f"""
    SELECT year, month, day, hour, count(*) as count
    FROM {db_name}.{schema_name}.{table_name}
    WHERE year = '2026' AND month = '02' AND day = '28' AND hour = '15'
    GROUP BY year, month, day, hour
""")
result_filtered.show()

+----+-----+---+----+-----+
|year|month|day|hour|count|
+----+-----+---+----+-----+
|2026|   02| 28|  15|    8|
+----+-----+---+----+-----+



In [41]:
# Exécuter une requête SQL avec un filtre sur les partitions de la table Iceberg
result_partition_filtered = spark.sql(f"""
    SELECT year, month, day, hour, count(*) as count
    FROM {db_name}.{schema_name}.{table_name}
    WHERE year = '2026' AND month = '02' AND day = '28' AND hour = '15'
    GROUP BY year, month, day, hour
""")
result_partition_filtered.show()

+----+-----+---+----+-----+
|year|month|day|hour|count|
+----+-----+---+----+-----+
|2026|   02| 28|  15|    8|
+----+-----+---+----+-----+



In [17]:
#==== Stopper la session Spark=====
spark.stop()